# 01 — Environment Setup & Data Loading

**Pre-condition:** Run `prepare_data.py` locally before this notebook.
It converts BDD100K into filtered YOLO/COCO splits and writes them to
`data_prepared/`. Upload that folder to Google Drive once.

This notebook:
1. Installs dependencies
2. Verifies the GPU environment
3. Mounts Drive and copies the pre-converted splits to `/content/`
4. Validates split image/label counts

In [1]:
import torch

print('Python:', __import__('sys').version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q ultralytics rfdetr fvcore nvidia-ml-py python-dotenv 
!pip install -q /content/drive/MyDrive/FON/master_rad/python_packages/thesis_src-0.1.0-py3-none-any.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.2/253.2 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 140.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 28.7 MB/s eta 0:00:00


In [4]:
from pathlib import Path
import os, zipfile

DRIVE_ROOT   = Path('/content/drive/MyDrive/FON/master_rad/bdd100k_data')
DATA_ROOT    = Path('/content/data_prepared')
YOLO_DATA_DIR = DATA_ROOT / 'yolo'
COCO_DATA_DIR = DATA_ROOT / 'coco'
CONFIGS_DIR   = DATA_ROOT / 'configs'

ZIP_SRC = DRIVE_ROOT / 'bdd100k_data.zip'

if not YOLO_DATA_DIR.exists():
    print('Extracting...')
    with zipfile.ZipFile(ZIP_SRC) as z:
        z.extractall('/content/')
    print('Done.')
else:
    print('Already extracted — skipping.')

Extracting...
Done.


In [5]:
import json

conditions = json.loads((CONFIGS_DIR / 'conditions.json').read_text())
splits = ['clear_day/train', 'clear_day/val'] + conditions['adverse']

print(f'{"Split":<30} {"Images":>8} {"Labels":>8} {"OK":>4}')
print('-' * 55)
for split in splits:
    img_dir = YOLO_DATA_DIR / split / 'images'
    lbl_dir = YOLO_DATA_DIR / split / 'labels'
    n_img = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    ok = '✓' if n_img > 0 and n_img == n_lbl else '✗'
    print(f'{split:<30} {n_img:>8} {n_lbl:>8} {ok:>4}')

Split                            Images   Labels   OK
-------------------------------------------------------
clear_day/train                   12454    12454    ✓
clear_day/val                      1764     1764    ✓
clear_dawn_dusk                    2311     2311    ✓
clear_night                        3274     3274    ✓
overcast_dawn_dusk                 1329     1329    ✓
overcast_daytime                   8590     8590    ✓
partly_cloudy_dawn_dusk             665      665    ✓
partly_cloudy_daytime              4900     4900    ✓
rainy_dawn_dusk                     383      383    ✓
rainy_daytime                      2918     2918    ✓
rainy_night                        2494     2494    ✓
snowy_dawn_dusk                     510      510    ✓
snowy_daytime                      3284     3284    ✓
snowy_night                        2522     2522    ✓
